In [126]:
import os
import glob
import pyabf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [171]:
def detect_pulse_onset(abf, threshold_mV_drop=0):
    """
    Detects pulse onset time from sweepC in sweep 0 (in seconds).
    """
    abf.setSweep(0)
    stim = abf.sweepC
    time = abf.sweepX

    baseline = np.median(stim[:100])
    delta = stim - baseline
    pulse_indices = np.where(delta < -threshold_mV_drop)[0]

    if pulse_indices.size == 0:
        return None

    onset_idx = pulse_indices[0]
    return time[onset_idx]  # seconds

def get_resistances(abf_path, rs_window_ms=20, rm_window_ms=(40, 49)):
    """
    For a given ABF file, compute:
    - Series resistance (max ΔI in first 20 ms after pulse)
    - Membrane resistance (mean I from 40–49 ms after pulse)
    - Noise (std dev in 40–49 ms window)
    Returns:
    - A list of dicts with sweep-level data
    """
    abf = pyabf.ABF(abf_path)

    if abf.sweepUnitsY != "pA":
        raise ValueError(f"{abf.abfID}: Not a voltage clamp file (unit: {abf.sweepUnitsY})")

    pulse_time_sec = detect_pulse_onset(abf)
    if pulse_time_sec is None:
        print(f"{abf.abfID}: No pulse detected.")
        return None

    results = []

    for sweep in range(abf.sweepCount):
        abf.setSweep(sweep)
        current = abf.sweepY
        time = abf.sweepX

        # --- Series Resistance: max delta in 0–20 ms after pulse ---
        rs_start = pulse_time_sec
        rs_end = pulse_time_sec + rs_window_ms / 1000
        rs_idx = np.where((time >= rs_start) & (time <= rs_end))[0]
        rs_segment = current[rs_idx] if rs_idx.size > 0 else None

        delta_I_rs = None
        series_resistance_MOhm = None
        if rs_segment is not None and len(rs_segment) > 0:
            delta_I_rs = np.max(rs_segment) - np.min(rs_segment)
            if delta_I_rs != 0:
                series_resistance_MOhm = 5e6 / delta_I_rs  # ΔV = -5 mV

        # --- Membrane Resistance: steady-state I in 40–49 ms ---
        rm_start = pulse_time_sec + rm_window_ms[0] / 1000
        rm_end = pulse_time_sec + rm_window_ms[1] / 1000
        rm_idx = np.where((time >= rm_start) & (time <= rm_end))[0]
        rm_segment = current[rm_idx] if rm_idx.size > 0 else None

        rm_baseline = np.mean(current[rs_idx]) if rs_idx.size > 0 else None
        rm_steady = np.mean(rm_segment) if rm_segment is not None and len(rm_segment) > 0 else None

        membrane_resistance_MOhm = None
        membrane_noise_pA = None
        if rm_baseline is not None and rm_steady is not None:
            delta_I_rm = rm_steady - rm_baseline
            if delta_I_rm != 0:
                membrane_resistance_MOhm = 5e6 / delta_I_rm
        if rm_segment is not None and len(rm_segment) > 0:
            membrane_noise_pA = np.std(rm_segment)

        results.append({
            "file": abf.abfID,
            "sweep": sweep,
            "max current change (pA)": delta_I_rs,
            "series resistance (MΩ)": series_resistance_MOhm,
            "membrane resistance (MΩ)": membrane_resistance_MOhm,
            "membrane noise (pA)": membrane_noise_pA
        })

    return results

In [173]:
def process_abf_directory_resistances(abf_dir, output_filename="resistances.csv"):
    """
    Process all ABF files in a directory to extract both series and membrane resistance.

    Saves output CSV in the same directory.
    """
    abf_files = glob.glob(os.path.join(abf_dir, "*.abf"))
    all_data = []

    for abf_path in abf_files:
        try:
            data = get_resistances(abf_path)
            if data:
                all_data.extend(data)
        except Exception as e:
            print(f"Error processing {abf_path}: {e}")

    df = pd.DataFrame(all_data)
    output_path = os.path.join(abf_dir, output_filename)
    df.to_csv(output_path, index=False)
    print(f"Saved results to {output_path}")
    return df

In [175]:
df = process_abf_directory_resistances("/Users/jayashri/Desktop/ManualGranuleRawABF")

Saved results to /Users/jayashri/Desktop/ManualGranuleRawABF/resistances.csv


In [177]:
def plot_resistance_and_noise_subplots(df, output_dir, threshold=0.20):
    """
    For each ABF file, plot three subplots:
    1. Series resistance
    2. Membrane resistance
    3. Membrane noise (std dev)
    Marks red stars where Rs or Rm deviate ≥20% from sweep 0.
    """
    for abf_id in df["file"].unique():
        sub_df = df[df["file"] == abf_id].dropna(subset=[
            "series resistance (MΩ)", "membrane resistance (MΩ)", "membrane noise (pA)"
        ])
        if sub_df.empty or sub_df.shape[0] < 2:
            continue

        sweeps = sub_df["sweep"].values
        rs = sub_df["series resistance (MΩ)"].values
        rm = sub_df["membrane resistance (MΩ)"].values
        noise = sub_df["membrane noise (pA)"].values

        # Deviation detection (≥20%)
        rs0, rm0 = rs[0], rm[0]
        rs_dev = np.abs(rs - rs0) / rs0
        rm_dev = np.abs(rm - rm0) / rm0
        rs_flags = np.where(rs_dev >= threshold)[0]
        rm_flags = np.where(rm_dev >= threshold)[0]

        # Set up subplots
        fig, axs = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

        # --- Series Resistance Plot ---
        axs[0].plot(sweeps, rs, 'o-', color='black', label='Series Resistance')
        if rs_flags.size > 0:
            axs[0].plot(sweeps[rs_flags], rs[rs_flags], 'r*', markersize=12, label='>20% deviation')
        axs[0].set_ylabel("Series Resistance (MΩ)")
        axs[0].set_title(f"{abf_id} – Resistance and Noise")
        axs[0].legend()
        axs[0].grid(True)

        # --- Membrane Resistance Plot ---
        axs[1].plot(sweeps, rm, 'o-', color='blue', label='Membrane Resistance')
        if rm_flags.size > 0:
            axs[1].plot(sweeps[rm_flags], rm[rm_flags], 'r*', markersize=12, label='>20% deviation')
        axs[1].set_ylabel("Membrane Resistance (MΩ)")
        axs[1].legend()
        axs[1].grid(True)

        # --- Membrane Noise Plot ---
        axs[2].plot(sweeps, noise, 'o-', color='green', label='Membrane Noise (pA)')
        axs[2].set_xlabel("Sweep")
        axs[2].set_ylabel("Noise (pA)")
        axs[2].legend()
        axs[2].grid(True)

        # Save figure
        plt.tight_layout()
        save_path = os.path.join(output_dir, f"{abf_id}_resistance_noise_subplots.png")
        plt.savefig(save_path, dpi=300)
        plt.close()
        print(f"Saved plot: {save_path}")

In [179]:
plot_resistance_and_noise_subplots(df, "/Users/jayashri/Desktop/ManualGranuleRawABF")

Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25612002_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25529004_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25610006_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25610011_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25529002_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25521002_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25521006_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25521012_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25603010_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleRawABF/25603006_resistance_noise_subplots.png
Saved plot: /Users/jayashri/Desktop/ManualGranuleR